### 08 - Conclusion et Recommandations
#### HumanForYou - Attrition ML

Ce notebook constitue la **synthese finale** du projet. Il consolide les resultats des 7 notebooks precedents, formule des recommandations actionnables pour HumanForYou, identifie les limites et propose des pistes d'amelioration.

---

## 1. Synthese du projet

### Pipeline global

| # | Notebook | Role | Sortie cle |
|---|---|---|---|
| 01 | EDA | Analyse exploratoire, qualite des donnees, hypotheses | `eda_summary.csv` |
| 02 | Preprocessing | Fusion, imputation, encodage Attrition | `attrition_merged_base.csv` |
| 03 | Feature Engineering | Variable `avg_work_hours` depuis la badgeuse | `attrition_with_avg_hours.csv` |
| 04 | KMeans Clustering | Segmentation non supervisee, profils employes | `kmeans_clusters.csv` |
| 05 | Preparation | Split, encodage, normalisation | `train/test_prepared.csv` |
| 06 | Classification | 7 modeles entraines, metriques, erreurs | Metriques + predictions |
| 07 | Comparaison | Cross-validation, choix modele final | `model_comparison.csv` |
| 08 | **Conclusion** | **Synthese, recommandations, limites** | Ce notebook |

### Donnees
- **4 410 employes**, 4 sources de donnees (RH, survey employe, survey manager, badgeuse)
- **26+ variables** apres feature engineering (incluant `avg_work_hours` et `cluster`)
- Qualite : coherence parfaite des IDs, < 1 % de NA sur variables RH, ~9,5 % sur badgeuse (conges)

In [1]:
# Chargement des resultats pour la synthese.
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join('..', 'data', 'processed')

# Metriques de comparaison des modeles
comparison_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'attrition_model_comparison.csv'))
cv_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'attrition_model_cv_scores.csv'))
kmeans_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'kmeans_clusters.csv'))

# Taux d'attrition
attrition_rate = kmeans_df['Attrition'].mean() * 100
n_employees = len(kmeans_df)
n_attrition = kmeans_df['Attrition'].sum()

# Meilleur modele
best = comparison_df.sort_values('f1_test', ascending=False).iloc[0]

print(f'Employes : {n_employees}')
print(f'Taux d\'attrition : {attrition_rate:.1f}% ({n_attrition} departs)')
print(f'Meilleur modele (F1 test) : {best["model"]} — F1={best["f1_test"]:.3f}, AUC={best["roc_auc_test"]:.3f}')
print(f'\nComparatif complet :')
display(comparison_df)

Employes : 4410
Taux d'attrition : 16.1% (711 departs)
Meilleur modele (F1 test) : RandomForest — F1=0.964, AUC=0.995

Comparatif complet :


,model,accuracy_test,precision_test,recall_test,f1_test,roc_auc_test
0,RandomForest,0.988662,1.000000,0.929577,0.963504,0.994903
1,SVM,0.913076,0.888889,0.525822,0.660767,0.935571
2,NaiveBayes,0.825397,0.451613,0.394366,0.421053,0.762594
3,Perceptron,0.812547,0.410256,0.375587,0.392157,0.734581
4,LogisticRegression,0.855631,0.622222,0.262911,0.369637,0.809237
5,KNN,0.831444,0.460938,0.276995,0.346041,0.874861
6,DecisionTree,0.851096,0.722222,0.122066,0.208835,0.728973


---

## 2. Resultats cles

### 2.1 Attrition
- **Taux global** : ~16 % (711 departs / 4 410 employes)
- **Cout estime** : a 50-200 % du salaire annuel par depart, cela represente un enjeu financier majeur
- **Desequilibre** : ratio ~5:1, necessitant des metriques adaptees (F1, AUC plutot qu'accuracy)

### 2.2 Facteurs d'attrition identifies (EDA)

| Facteur | Signal | Force |
|---|---|---|
| **BusinessTravel** | Deplacements frequents → taux d'attrition plus eleve | Fort |
| **MonthlyIncome** | Salaire bas → plus de departs | Fort |
| **Age** | Employes jeunes (< 30 ans) plus volatils | Modere |
| **YearsAtCompany** | Anciennete < 3 ans = zone critique | Fort |
| **MaritalStatus** | Celibataires ~25 % vs maries ~13 % | Modere |
| **JobRole** | Sales Rep, Lab Technician a risque | Modere |

### 2.3 Clustering
- Segmentation KMeans revelant des **profils d'employes distincts** (anciennete, salaire, age)
- Taux d'attrition **significativement different** entre clusters → segmentation pertinente
- La feature `cluster` enrichit les modeles de classification

### 2.4 Modele de classification retenu
- **7 modeles compares** : Logistic Regression, Perceptron, SVM, KNN, Naive Bayes, Decision Tree, Random Forest
- **Cross-validation 5-fold** : scores stables, faible risque d'overfitting
- **Modele retenu** : selectionne par score composite (F1 test + AUC + stabilite CV)
- Le modele est capable de discriminer les profils a risque avec un compromis precision/recall adapte au contexte RH

---

## 3. Recommandations pour HumanForYou

### 3.1 Actions immediates (court terme)

| # | Recommandation | Justification | Priorite |
|---|---|---|---|
| 1 | **Limiter les deplacements professionnels frequents** | BusinessTravel = facteur d'attrition significatif | Critique |
| 2 | **Revoir la politique salariale** des postes a risque (Sales Rep, Lab Tech) | Salaire bas = signal fort de depart | Haute |
| 3 | **Programme d'integration renforce** pour les 3 premieres annees | YearsAtCompany < 3 = zone critique | Haute |
| 4 | **Cibler les profils juniors/celibataires** pour les programmes de retention | Population la plus volatile | Moyenne |

### 3.2 Actions structurelles (moyen terme)

| # | Recommandation | Justification |
|---|---|---|
| 5 | **Monitorer le cluster a haut risque** identifie par le KMeans | Permet un ciblage proactif |
| 6 | **Deployer le modele comme outil d'aide a la decision** (jamais en automatique) | Alertes sur groupes a risque |
| 7 | **Former les managers** a l'interpretation des indicateurs | Eviter les dérives d'usage |
| 8 | **Mettre en place un suivi longitudinal** | Donnees actuelles = photo statique |

### 3.3 Garde-fous ethiques
- **Aucun scoring individuel** communique → uniquement des tendances collectives
- **Revue humaine obligatoire** avant toute action basee sur le modele
- **Audit de fairness** regulier (cf. livrable ethique)
- **Transparence** : les employes doivent etre informes de l'existence du dispositif

---

## 4. Limites du projet

| Limite | Impact | Mitigation |
|---|---|---|
| **Donnees statiques** (snapshot 2015) | Pas de dynamique temporelle, pas d'evolution | Collecte longitudinale pour le futur |
| **Correlation ≠ causalite** | Les facteurs identifies ne sont pas forcement des causes | Validation par experts RH, A/B testing |
| **Silhouette moderee** (clustering) | Clusters partiellement chevauchants | Normal pour des donnees RH, completer avec d'autres methodes |
| **Variables manquantes** | Pas de satisfaction salaire, mobilite interne, formation | Enrichir les sources de donnees |
| **Desequilibre de classes** (84/16) | Biais vers la classe majoritaire | SMOTE, ajustement du seuil de decision |
| **Petit effectif sur certains sous-groupes** | HR (189), EducationField HR (81) | Resultats fragiles statistiquement sur ces groupes |
| **Biais de sélection QVT** | Non-reponse survey = potentiellement les moins satisfaits | Sensibiliser a la completude des surveys |
| **Pas de validation externe** | Pas de donnees d'une autre periode ou entreprise | Deployer en mode pilote avec monitoring |

---

## 5. Ameliorations futures

### 5.1 Donnees
- **Donnees temporelles** : suivi mensuel/trimestriel pour capter les evolutions (satisfaction, performance)
- **Sources supplementaires** : entretiens annuels, demandes de formation, mobilite interne, absenteisme
- **Donnees textuelles** : analyse de sentiment sur les verbatims des enquetes QVT

### 5.2 Modelisation
- **Gradient Boosting** (XGBoost, LightGBM, CatBoost) : generalement superieurs aux modeles testes
- **Hyperparameter tuning** : GridSearchCV / RandomizedSearchCV / Optuna
- **SMOTE / ADASYN** : reequilibrage de la classe minoritaire
- **Feature selection** : RFE, SelectFromModel, elimination de la multicolinearite
- **Stacking / Blending** : combinaison de modeles pour robustesse

### 5.3 Explicabilite
- **SHAP values** : explication globale et locale des predictions
- **LIME** : explications par approximation locale
- **Partial Dependence Plots** : effet marginal de chaque variable

### 5.4 Deploiement
- **API REST** (Flask/FastAPI) pour integration aux outils SIRH
- **Dashboard** : monitoring des predictions et des metriques en temps reel
- **Retraining** : pipeline automatise de reentrainement periodique
- **A/B testing** : valider l'impact des actions de retention recommandees

---

## 6. Structure de presentation (soutenance)

### Plan slides recommande

**Slide 1 — Contexte & Probleme RH**
- HumanForYou : 4 410 employes, pharma
- Taux d'attrition : ~16 % → enjeu financier et humain
- Objectif : identifier les facteurs, anticiper les departs

**Slide 2 — Donnees & Pipeline**
- 4 sources : RH, surveys, badgeuse
- Pipeline 01→08 : EDA → Preprocessing → Feature Eng → Clustering → Classification → Conclusion
- Qualite : coherence IDs, gestion NA

**Slide 3 — EDA : Insights cles**
- Distributions : profils jeunes/peu anciens surrepresentes dans les departs
- Facteurs discriminants : BusinessTravel, salaire, anciennete
- Correlations faibles → probleme multivarié

**Slide 4 — Clustering : Profils employes**
- KMeans : k selectionne via silhouette + elbow + Davies-Bouldin
- Profils distincts (juniors vs seniors)
- Taux d'attrition different par cluster → segmentation pertinente

**Slide 5 — Classification : Modele retenu**
- 7 modeles compares (Logistic, SVM, RF...)
- Cross-validation 5-fold : resultats stables
- Modele retenu : meilleur F1 + AUC + stabilite
- Matrice de confusion : analyse FP/FN

**Slide 6 — Ethique & Biais**
- 7 exigences UE couvertes
- Choix ethique B+D : modele explicable + communication agregee
- Biais identifies : MaritalStatus, petits groupes
- Garde-fous : pas de scoring individuel, audit fairness

**Slide 7 — Recommandations client**
- 4 actions immediates (BusinessTravel, salaires, integration, ciblage)
- 4 actions structurelles (monitoring, formation managers)
- Deploiement en aide a la decision uniquement

**Slide 8 — Limites & Perspectives**
- Donnees statiques, correlation ≠ causalite
- Pistes : XGBoost, SHAP, donnees temporelles, A/B testing

---

### Conseils pour la soutenance
- **Chaque choix doit etre justifie** : "Nous avons choisi KMeans parce que..."
- **Montrer qu'on connait les limites** : "Le silhouette score est modere, ce qui est attendu en RH"
- **Distinguer correlation et causalite** : "Nous avons identifie des associations, pas des causes"
- **Relier chaque resultat a un impact metier** : "Le recall est critique car un FN = un depart non anticipe"
- **Evoquer l'ethique spontanement** (pas seulement quand on le demande)

---

## Conclusion finale

Ce projet demontre une demarche complete de data science appliquee aux RH :

1. **Exploration rigoureuse** : qualite des donnees validee, hypotheses formulees, visualisations interpretees
2. **Segmentation pertinente** : le clustering revele des profils avec des taux d'attrition significativement differents
3. **Classification performante** : 7 modeles compares, cross-valides, avec un choix argumente
4. **Ethique integree** : les 7 exigences UE sont couvertes, les biais identifies, les garde-fous proposes
5. **Recommandations actionnables** : des leviers concrets pour reduire l'attrition chez HumanForYou

Le modele retenu est un **outil d'aide a la decision**, pas un automate. Son deploiement doit s'accompagner de formation, de monitoring et de revue humaine systematique.

---

*Projet realise dans le cadre de la formation CESI — Bloc Intelligence Artificielle*